# Smoke Test — the go/no-go gate

**Asch in Silicon** · [repo](https://github.com/LihanCanCode/Collective-Cognitive-Error)

Answers the one question that can kill the project before anything is built on top of it:
**do these models conform to a wrong majority at a measurable rate at all?**

### Before you run

Kaggle sidebar → **Settings**:
- **Accelerator:** `GPU T4 x2`
- **Internet:** `On`  (required — needs to clone the repo and download the model)

Expect **10–20 min**, most of it model download.

### Reading the result

| Signal | Meaning |
|---|---|
| baseline error at n=0 **< 10%** | items are easy enough that a wrong answer under pressure means conformity, not ignorance |
| conformity rate **5–70%** | **PASS** — measurable band, proceed to the full grid |
| conformity **~0%** | items too easy or model too independent → raise difficulty |
| conformity **~100%** | no independent judgement left → lower difficulty or change model |

**Do not skip Cell 5.** The conformity number is meaningless if confederates broke character or the
naive agent answered in a format the parser missed — and that failure looks exactly like a clean result.

## Cell 1 — install

Kaggle ships a preinstalled torch that sometimes conflicts with vLLM. If the import in Cell 4 fails,
**restart the session once** (Run → Restart session) and re-run from Cell 2 — the install persists.

In [ ]:
!pip install -q vllm==0.6.3
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Cell 2 — get the code, verify it offline

The mock run is a full end-to-end pass with no GPU and no network. If it does not print `PASS`,
something is wrong with the harness itself and no GPU time should be spent.

In [ ]:
import os

REPO = "https://github.com/LihanCanCode/Collective-Cognitive-Error.git"

!rm -rf /kaggle/working/repo
!git clone -q $REPO /kaggle/working/repo
os.chdir("/kaggle/working/repo")
print("cwd:", os.getcwd())

!python -m scripts.make_smoke_bank
!python -m scripts.run_smoke --backend mock --out results/mock_check.jsonl

## Cell 3 — run the gate

50 items × {n=0 control, n=3 unanimous-wrong} = 100 trials. n=3 is where human conformity peaks in
Asch (32%).

Resumable: if the session dies, just re-run this cell — completed trials are skipped.

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"
RESULTS = "results/smoke_qwen7b.jsonl"

!python -m scripts.run_smoke --backend vllm --model $MODEL --out $RESULTS

## Cell 4 — read five transcripts by hand

Not optional. You are checking three things:
1. Did each confederate actually assert its assigned answer? (`complied=True`)
2. Did the naive agent answer in the `Answer: X` format? (`valid=True`)
3. Does the naive agent's reasoning look like real deliberation, or is it degenerate?

In [ ]:
import json
from pathlib import Path

records = [json.loads(line) for line in Path(RESULTS).open() if line.strip()]
critical = [r for r in records if r["n_confederates"] == 3]
print(f"{len(records)} trials total, {len(critical)} critical\n")

for rec in critical[:5]:
    print("=" * 95)
    print(
        f"stance={rec['stance']}  answer={rec['answer']}  correct={rec['correct_answer']}  "
        f"majority={rec['majority_answer']}  valid={rec['valid']}"
    )
    for turn in rec["transcript"]:
        who = turn["role"]
        extra = (
            f" (assigned {turn['assigned_answer']}, complied={turn['complied']})"
            if who == "confederate"
            else ""
        )
        print(f"\n--- {who}{extra} ---")
        print(turn["text"][:400])

## Cell 5 — diagnostics

If the verdict was FAIL, these numbers say which knob to turn. **Report these back.**

In [ ]:
from collections import Counter

n = max(len(critical), 1)
break_rate = sum(1 for r in critical if not r.get("confederates_complied")) / n
parse_fail = sum(1 for r in critical if r.get("answer") is None) / n
errors = Counter(r["error"].split(":")[0] for r in records if r.get("error"))

print("stance distribution: ", dict(Counter(r["stance"] for r in critical)))
print(f"confederate break rate: {break_rate:.1%}")
print(f"parse failure rate:     {parse_fail:.1%}")
print("runtime errors:      ", dict(errors) or "none")

print("""
How to read this
----------------
High break rate     -> confederates refuse to argue for the wrong answer. Soften the confederate
                       instruction, or use a less safety-tuned confederate model.
High parse failure  -> the model ignores the output format. Tighten the format instruction or add
                       a one-shot example. Do NOT loosen the parser to compensate -- that would
                       manufacture answers the model never gave.
CR ~0, both low     -> genuine independence. Raise item difficulty (Asch: conformity rises with
                       difficulty) rather than concluding the effect is absent.
""")

## Cell 6 — save results off the session

Kaggle wipes everything outside `/kaggle/working` when the session ends. Either **Save Version**
(persists `/kaggle/working` as notebook output) or download the JSONL from the file browser.

In [ ]:
import shutil
from pathlib import Path

dest = Path("/kaggle/working") / Path(RESULTS).name
shutil.copy(RESULTS, dest)
print(f"saved -> {dest}  ({dest.stat().st_size / 1024:.0f} KB)")

## Optional — try a second model

If Qwen-7B fails the gate, the fastest diagnostic is a second model: a model-specific quirk looks
very different from a genuine ceiling in the item bank. All ungated, no HF token needed.

Change `MODEL` and `RESULTS` in Cell 3, then re-run Cells 3–6.

In [ ]:
# mistralai/Mistral-7B-Instruct-v0.3
# google/gemma-2-9b-it
# Qwen/Qwen2.5-1.5B-Instruct    <- smaller: expect MORE conformity if the effect is real

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RESULTS = "results/smoke_qwen1_5b.jsonl"

!python -m scripts.run_smoke --backend vllm --model $MODEL --out $RESULTS